In [ ]:
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline

# 1. Zabalíme Matějův kód do profi Scikit-learn formátu
class HolmanImpute(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.stats = {}

    def fit(self, X, y=None):
        # Model se zde UČÍ a pamatuje si statistiky z trénovacích dat
        self.stats['income_medians'] = X.groupby('Parental_Education')['Family_Income'].median()
        self.stats['study_means'] = X.groupby('Parental_Education')['Study_Hours_per_Day'].mean()
        self.stats['stress_median'] = X['Stress_Index'].median()
        self.stats['age_mean'] = X['Age'].mean()
        return self

    def transform(self, X):
        X_copy = X.copy()

        X_copy['Family_Income'] = X_copy['Family_Income'].fillna(X_copy['Parental_Education'].map(self.stats['income_medians']))
        X_copy['Study_Hours_per_Day'] = X_copy['Study_Hours_per_Day'].fillna(X_copy['Parental_Education'].map(self.stats['study_means']))
        X_copy['Stress_Index'] = X_copy['Stress_Index'].fillna(self.stats['stress_median'])

        X_copy['Age_Gap'] = X_copy['Age'] - self.stats['age_mean']

        cols_to_drop = ['Age', 'Parental_Education'] 
        cols_to_drop = [c for c in cols_to_drop if c in X_copy.columns]
        X_copy = X_copy.drop(columns=cols_to_drop)

        return X_copy